# 05 - The photon transfer curve

**Purpose.** Run `protocols/02-ptc.md` and publish `g(gain)` - electrons per ADC count, per CFA
plane, at nine gain settings - together with the gain law fitted through it and the FPN test that
says whether sigma^2 is shot plus read and nothing else. It is the session that puts a scale on
every count session 01 measured.

**What it is not for.** Linearity, `ceiling(gain)` and full well: those need a characterised
light source, a shrunk ROI and a per-channel bend, and are a later session. Nor dark current.
Nothing here fits a bend, and no rung is placed to find one.

**This notebook captures; `06_ptc_read` explains.** The reading half publishes to `results/`.
Everything below either talks to the camera or decides what to ask it for next.

## 1. Pre-flight, and the attenuation scout

**This section is `light-source.md` item 3, and it runs *warm*, before and apart from the session
proper.** It answers the one question the protocol cannot answer on paper: *what grey level and
how many sheets* put the bench where a twelve-rung ladder fits between the camera's shortest
exposure and a sane wall clock.

It is deliberately not gate 3. Gate 3 solves `t_sat(gain)` from the measured flux *inside* the
session, cold, with the bench undisturbed, and that is what the record quotes. This scout produces
a **bench configuration** - a sheet count and a grey level - plus a prediction of `t_sat` that
gate 3 re-measures. Nothing here is published to `results/`.

**Why it may run warm.** Flux is an optical measurement and the sensor's temperature does not
change how much light arrives. What temperature does change - dark current - is held down by
keeping the scout's exposures short, and none of its numbers survive into the record.

### The arithmetic that sets the target

Gain is in 0.1 dB units, so amplification is `10 ** (gain/200)` and **gain 450 is 178x gain 0**.
(600 would have been 1000x, and is out of this project's scope - `CLAUDE.md`, D52. Dropping it is
what brings the attenuation this bench needs from ~4000x down to ~800x.) With one fixed light
level `t_sat` spans 178:1 across the gain set, and the ladder spans another 300:1 (0.3% to 90%),
so the session's shortest exposure is

    0.003 * t_sat(450)  =  1.69e-5 * t_sat(0)

so `t_sat(0)` alone fixes the whole session: the margin over both floors below, and about
`25 * t_sat(0)` of shutter-open time.

**There are two floors, and the obvious one does not bind.** The camera's shutter stops at 32 us;
the panel's refresh period is 16.7 ms, **520x longer**. An exposure shorter than one period sees
whichever slice of the refresh cycle it lands in, and on a rolling-shutter sensor that slice
differs row by row - so the error arrives as spatial structure, which does *not* cancel in the
pair difference the way a uniform level drift does.

| `t_sat(0)` | faintest rung | margin over 32 us | rungs under one refresh period | session exposure |
|---|---|---|---|---|
| 1.9 s | 31 us | 1.0x | 55 of 96 | 0.6 min |
| 5 s | 81 us | 2.5x | 40 of 96 | 1.6 min |
| 12 s | 194 us | 6.1x | 27 of 96 | 3.9 min |
| 20 s | 323 us | 10.1x | 21 of 96 | 6.5 min |
| 30 s | 485 us | 15.2x | 17 of 96 | 9.7 min |

**Target: `t_sat(0)` between 10 and 30 s, aimed at the top of that window.** Wall clock is the
only thing the long end costs, and ~10 minutes of shutter-open against 464 frames of download and
save is not what makes this session long. Every other column improves.

**No configuration clears the refresh floor, which is why the flicker test below is a gate and
not a curiosity.** Emptying the last column would take `t_sat(0)` near 1000 s - seven hours of
shutter-open - so the high gains shoot sub-refresh rungs whatever we choose, and the question is
not how to avoid them but whether this panel is steady enough to make them harmless. An LCD holds
a static grey between refreshes and at 100% brightness its backlight is usually driven DC, in
which case exposure length is irrelevant and 0.5 ms is as good as 5 s. That is a claim about
*this* iPad, and the test measures it rather than assuming it.

**One level for all eight gains, not a level per gain.** The PTC plots variance against measured
signal, so re-attenuating between gains would be perfectly legitimate - but a fixed source makes
`t_sat(gain) * amplification(gain)` a constant, and that is a check on the 0.1 dB law from
timings alone, before a single variance is computed. Give it up only if the flicker test says the
high-gain rungs are unusable; the fallback is then to dim the panel for the top gains, paying
that check and a second flux in the record.


In [ ]:
import json
import pathlib
import sys
import time
import urllib.request

import numpy as np

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import asi, spatial, stats

RESULTS = pathlib.Path.cwd().parent / "results"

FULL_SCALE = 4095                    # ADC counts; the units rule in CLAUDE.md
GAINS = [0, 50, 100, 190, 200, 250, 300, 450]   # 450 is the ceiling (D52)
RUNGS = [0.3, 0.5, 0.85, 1.4, 2.4, 4.0, 6.8, 11.4, 19.2, 32, 54, 90]   # % of t_sat
ROI = (1408, 568, 1024, 1024)        # even origin and extent, or Bayer shifts (L05)
OFFSET = 15                          # project_offset, fixed by session 01
SCOUT_GAIN = 100
MAX_EXPOSURE = 2.0                   # s; the scout never needs a long frame

# The bench configuration this run measures.  A flux with no configuration
# beside it is not a measurement of anything (light-source.md item 3).
SHEETS = 8                           # sheets of paper between camera and panel
REFRESH_HZ = 60.0                    # panel refresh; one period is the flicker yardstick
PATCH_SERVER = "http://127.0.0.1:8765"

TARGET_TSAT0 = (10.0, 30.0)          # s, the window the table above argues for
AIM_TSAT0 = 30.0                     # aim at the long end: only wall clock pays for it

_bias = json.loads((RESULTS / "bias_constants.json").read_text())
PEDESTAL_FIT = _bias["pedestal_fit"]["value"]
MIN_EXPOSURE = _bias["bias_exposure"]["value"]
HCG = _bias["hcg_threshold_gain"]["value"]


def amplification(gain):
    """Gain is in 0.1 dB units, so 200 units is exactly a factor of ten."""
    return 10.0 ** (gain / 200.0)


def pedestal(gain):
    """Published pedestal at offset 15, in ADC counts (03_bias_sweep)."""
    branch = PEDESTAL_FIT["hcg" if gain >= HCG else "lcg"]
    return branch["A"] + branch["B"] * amplification(gain)


def plane_means(mosaic):
    """Mean of each CFA plane, in ADC counts.  `to_adc` raises rather than
    truncate, so this doubles as a check that the frame came off the raw path."""
    return {k: float(stats.to_adc(v).mean()) for k, v in spatial.split(mosaic).items()}


def set_level(level, settle_s=0.8):
    """Drive `grey-patch.html` from here (protocols/patch-server.py).

    The page polls and applies the change, so the settle covers one poll plus a
    repaint.  Whether the panel actually followed is not taken on trust: the
    sweep below reads it back out of the pixels.
    """
    with urllib.request.urlopen(f"{PATCH_SERVER}/set?level={int(level)}", timeout=5) as r:
        state = json.load(r)
    time.sleep(settle_s)
    return state


print(f"bench: {SHEETS} sheets of paper, grey level driven from here")
print(f"min exposure {MIN_EXPOSURE * 1e6:.0f} us")
print("pedestal at offset 15:  " + "  ".join(f"g{g}={pedestal(g):.0f}" for g in GAINS))

### Gate 1 - white balance, verified from the pixels (L01)

The camera ships `WB_R=55`, `WB_B=75` and applies them to RAW16 before the data reaches us. The
control reading back as 50 proves only that the control took. The evidence is the modal step
between adjacent distinct values: **16 on all four planes**, greens at 16 with red 17/18 and blue
24 being the fingerprint of white balance still applied.

Nothing captured before this passes is usable - the scout included, because a smeared step means
`to_adc` refuses and every mean below is wrong.

In [ ]:
rig = asi.open_camera()
asi.neutralise_white_balance(rig)
asi.configure(rig, gain=SCOUT_GAIN, offset=OFFSET, roi=ROI)

dark, hdr = asi.capture(rig, MIN_EXPOSURE, imagetyp="DARK")
steps = {name: stats.value_step(p) for name, p in spatial.split(dark).items()}

print("modal value step per plane:", steps)
print("WB_R", rig.get("WB_R"), " WB_B", rig.get("WB_B"),
      " gain", rig.get("Gain"), " offset", rig.get("Offset"))
assert set(steps.values()) == {16}, f"gate 1 FAILED: {steps} -- stop, do not correct later"
print("\ngate 1 passed")

### Measuring a flux

One helper, used by everything below: step the exposure until the brightest plane lands near
mid-scale, then read flux off as `(mean - pedestal) / exptime`.

Mid-scale rather than near full scale on purpose. A plane close to 4095 is compressed by whatever
non-linearity lives near saturation, and this session is explicitly not the one that measures a
bend; half scale keeps the estimate on the part of the curve we are entitled to call straight.
One frame is discarded after every exposure change, as the protocol requires.

In [ ]:
ped_scout = pedestal(SCOUT_GAIN)


def auto_expose(start=1e-3, tries=9):
    """Land the brightest plane between 35% and 65% of full scale.

    Returns `(means, exptime, converged)`.  A run that does not converge is
    still returned rather than raised on: at the dim end the honest outcome is
    "as long as this scout will go and still not bright", and that is data.
    """
    exposure, means, exptime = start, None, start
    for _ in range(tries):
        asi.capture(rig, exposure, imagetyp="LIGHT")          # discard after the change
        mosaic, h = asi.capture(rig, exposure, imagetyp="LIGHT")
        means, exptime = plane_means(mosaic), h["EXPTIME"]
        top = max(means.values())
        if 0.35 * FULL_SCALE <= top <= 0.65 * FULL_SCALE:
            return means, exptime, True
        scale = 0.5 * FULL_SCALE / max(top - ped_scout, 1.0)
        nxt = min(max(exposure * scale, MIN_EXPOSURE), MAX_EXPOSURE)
        if abs(nxt - exposure) / exposure < 0.02:             # pinned at a limit
            break
        exposure = nxt
    return means, exptime, False


def flux_of(means, exptime):
    return {k: (v - ped_scout) / exptime for k, v in means.items()}


set_level(255)
means, exptime, ok = auto_expose()
flux = flux_of(means, exptime)
bright = max(flux, key=flux.get)

print(f"anchor: grey level 255, {SHEETS} sheets, gain {SCOUT_GAIN}, "
      f"{exptime * 1e6:.0f} us, converged={ok}")
for k, v in flux.items():
    print(f"  {k}: {v:12.1f} counts/s   ({v / flux[bright] * 100:5.1f}% of {bright})")

### What that flux implies

`t_sat(gain)` is where the **brightest** plane fills the headroom above its own pedestal -
brightest, because that is the plane that clips first and clipping is what the top rung must
avoid. The pedestal is not a detail here: at gain 600 it is over 1000 counts, a quarter of full
scale. At gain 450, the top of this project's range, it is 231 counts and the headroom is 3864.

In [ ]:
def tsat_table(flux_ref, label=""):
    amp_ref = amplification(SCOUT_GAIN)
    print(f"{label}\n{'gain':>5} {'amp':>8} {'pedestal':>9} {'headroom':>9} "
          f"{'t_sat':>12} {'0.3% rung':>12} {'90% rung':>12}")
    out = {}
    for g in GAINS:
        head = FULL_SCALE - pedestal(g)
        t = head / (flux_ref * amplification(g) / amp_ref)
        out[g] = t
        print(f"{g:5d} {amplification(g):8.1f} {pedestal(g):9.1f} {head:9.1f} "
              f"{t:11.4g}s {RUNGS[0] / 100 * t * 1e6:11.4g}u {RUNGS[-1] / 100 * t:11.4g}s")

    t0, ttop = out[0], out[GAINS[-1]]
    shortest = RUNGS[0] / 100 * ttop
    print(f"\nt_sat(gain 0)        {t0:12.4g} s    (target "
          f"{TARGET_TSAT0[0]:.0f}-{TARGET_TSAT0[1]:.0f} s)")
    print(f"shortest rung        {shortest * 1e6:12.4g} us   (floor {MIN_EXPOSURE * 1e6:.0f} us)")
    print(f"session shutter-open {25.1 * t0 / 60:12.4g} min")
    want = sum(TARGET_TSAT0) / 2
    print(f"attenuation to reach t_sat(0) = {want:.0f} s: {want / t0:.4g}x")
    return out


tsat_table(flux[bright], f"bench as it stands (level 255, {SHEETS} sheets; "
                         f"{bright} sets t_sat):")

### The grey-level curve, measured rather than assumed

L07 says grey level is exhausted below about 25% of full scale, because the backlight leaks
through a black LCD. That is a claim from a retired attempt about a different bench, and it is
cheap to check here: the level is driven from this notebook, so the curve costs no trips to the
iPad.

Two things come out of it. The **usable range** - how much attenuation the level alone can
deliver before the leak floor - and a check that the panel is actually following: if the page is
not connected to the server, every level returns the same flux and the table below is flat.

In [ ]:
LEVELS = [255, 224, 192, 160, 128, 96, 64, 48, 32, 24, 16, 8, 0]

curve = []
for lv in LEVELS:
    set_level(lv)
    m, e, converged = auto_expose(start=max(exptime, MIN_EXPOSURE))
    curve.append((lv, flux_of(m, e)[bright], e, converged))

f255 = curve[0][1]
print(f"{'level':>6} {'% of white':>11} {'flux':>14} {'attenuation':>12} "
      f"{'exptime':>10} {'converged':>10}")
for lv, f, e, converged in curve:
    print(f"{lv:6d} {lv / 255 * 100:10.1f}% {f:14.1f} {f255 / max(f, 1e-9):11.4g}x "
          f"{e * 1e6:9.0f}u {str(converged):>10}")

best = curve[-1][1]
print(f"\ngrey level alone gives {f255 / max(best, 1e-9):.4g}x, from 255 down to 0.")
assert curve[-1][1] < 0.5 * f255, ("level 0 is as bright as level 255 -- the panel is not "
                                   "following the server.  Reload grey-patch.html on the iPad.")

### Choosing the configuration

The curve above is a set of measured fluxes; this is the one decision the scout exists to make.
Each level implies a `t_sat(0)`, and each `t_sat(0)` implies all 96 exposures the session would
shoot - so the choice is judged against the whole ladder rather than against a single number.

No interpolation. The level sweep steps by roughly 1.6x in flux, the target window is 3x wide,
and a level that was measured is worth more than one that was fitted. If nothing lands inside the
window the sheet stack is the wrong thickness: change it, re-run the scout, and **never** scale
the flux by a per-sheet figure (L07, L08 - stacked diffusers give diminishing returns).

The ladder it prints is capped at the top by `top_rung`: a rung whose own noise no longer fits
under 4095 measures a variance that is too low and a `g` that is too high, and the pair difference
cannot see through it. That is a protocol rule (`02-ptc.md`), found by this scout on 2026-08-31 -
gain 450 read `g = 0.0593` at 90% against 0.0511 at its clean rungs - and it moves two rungs: 90%
becomes 89% at gain 300 and **75% at gain 450**.


In [ ]:
period = 1.0 / REFRESH_HZ
N_RUNGS = len(GAINS) * len(RUNGS)
CLIP_SIGMAS = 4.0                     # of a frame's own noise, under the top code


def g_predicted(gain):
    """L25's gain anchor carried by L29's 0.1 dB slope, in e- per ADC count.

    A prediction, used to place a rung and nothing else.  It never enters a
    published number, and it errs safe: a larger true `g` only widens the margin
    the cap below is protecting.
    """
    return 9.382 * 10 ** (-0.00502 * gain)


def top_rung(gain):
    """The highest rung, in % of headroom, whose own noise still fits under 4095.

    Solve `S + k*sqrt(S/g) = headroom` for S.  Above it the bright tail of the
    frame's own spread is censored by the top code, the measured variance comes
    in low, and that reads as a `g` that is too high -- which pair-differencing
    cannot see through, because the clipping happens before the subtraction
    (protocols/02-ptc.md).  It binds only where `g` is small: 90% everywhere up
    to gain 250, 89% at 300, 75% at 450.
    """
    head = FULL_SCALE - pedestal(gain)
    gg = g_predicted(gain)
    s = (-CLIP_SIGMAS / np.sqrt(gg) + np.sqrt(CLIP_SIGMAS ** 2 / gg + 4 * head)) / 2
    return min(RUNGS[-1], 100 * s * s / head)


def ladder_for(t0):
    """Every exposure the session shoots, in seconds, keyed by gain.

    One light level and eight gains leaves no free parameter: `t_sat` scales as
    the headroom above that gain's pedestal, divided by the amplification.  The
    top rungs are capped by `top_rung`, so the ladder is 0.3% to 90% of `t_sat`
    only where 90% is a rung that can be measured.
    """
    head0 = FULL_SCALE - pedestal(0)
    out = {}
    for g in GAINS:
        t_sat = t0 * (FULL_SCALE - pedestal(g)) / head0 / amplification(g)
        cap = top_rung(g)
        out[g] = [min(r, cap) / 100 * t_sat for r in RUNGS]
    return out


def tsat0_of(flux_scout):
    """t_sat at gain 0 implied by a flux measured at SCOUT_GAIN."""
    return (FULL_SCALE - pedestal(0)) / (flux_scout / amplification(SCOUT_GAIN))


print(f"{'level':>6} {'t_sat(0)':>10} {'faintest':>11} {'x floor':>9} "
      f"{'sub-refresh':>13} {'in window':>10}")
choices = []
for lv, f, e, converged in curve:
    if f <= 0:
        continue
    t0 = tsat0_of(f)
    rungs = ladder_for(t0)
    shortest = min(min(v) for v in rungs.values())
    n_sub = sum(1 for v in rungs.values() for x in v if x < period)
    inside = TARGET_TSAT0[0] <= t0 <= TARGET_TSAT0[1]
    choices.append((lv, t0, inside))
    print(f"{lv:6d} {t0:9.4g}s {shortest * 1e6:10.4g}u {shortest / MIN_EXPOSURE:8.1f}x "
          f"{n_sub:8d}/{N_RUNGS:<4d} {str(inside):>10}")

usable = [c for c in choices if c[2]]
if usable:
    level, t_sat0, _ = min(usable, key=lambda c: abs(c[1] - AIM_TSAT0))
    print(f"\nbench configuration: grey level {level}, {SHEETS} sheets, "
          f"t_sat(0) = {t_sat0:.4g} s")
else:
    level, t_sat0, _ = min(choices, key=lambda c: abs(np.log(c[1] / AIM_TSAT0)))
    print(f"\nNo level lands in {TARGET_TSAT0[0]:.0f}-{TARGET_TSAT0[1]:.0f} s; closest is "
          f"level {level} at {t_sat0:.4g} s.")
    print(f"The stack is off by {AIM_TSAT0 / t_sat0:.3g}x in attenuation -- "
          f"{'add' if t_sat0 < AIM_TSAT0 else 'remove'} sheets, re-run the scout, and do not "
          f"predict the new flux from a per-sheet figure.")

flux_at = {lv: f for lv, f, _, _ in curve}
brighter = [lv for lv, _, _ in choices if lv > level]
if brighter and flux_at[min(brighter)] / flux_at[level] < 1.2:
    print(f"NOTE: level {level} is on the leak floor -- level {min(brighter)} is only "
          f"{flux_at[min(brighter)] / flux_at[level]:.2f}x brighter, so the level is not "
          f"driving the flux and there is no trim in this direction.  It is the backlight "
          f"leaking through a black LCD (L07), not a level anyone set.  The configuration "
          f"works; adding sheets is what moves it back onto the responsive part of the curve.")

lad = ladder_for(t_sat0)
head0 = FULL_SCALE - pedestal(0)
print(f"\n{'gain':>5} {'t_sat':>10} {'faintest rung':>15} {'top rung':>11} {'of t_sat':>9} "
      f"{'sub-refresh':>12} {'x shutter floor':>16}")
for g in GAINS:
    v = lad[g]
    t_sat = t_sat0 * (FULL_SCALE - pedestal(g)) / head0 / amplification(g)
    print(f"{g:5d} {t_sat:9.4g}s {v[0] * 1e3:14.4g}ms {v[-1]:10.4g}s {top_rung(g):8.0f}% "
          f"{sum(1 for x in v if x < period):9d}/{len(RUNGS):<2d} {v[0] / MIN_EXPOSURE:15.1f}x")

print(f"\nblock 1 shutter-open {sum(4 * sum(v) for v in lad.values()) / 60:.1f} min; "
      f"block 2's bias frames are {MIN_EXPOSURE * 1e6:.0f} us each and cost nothing.")
print(f"faintest rung of the session: {min(min(v) for v in lad.values()) * 1e6:.0f} us at gain "
      f"{GAINS[-1]}, {min(min(v) for v in lad.values()) / period:.3f} of a refresh period.")


### Does the panel flicker? - the gate on the high-gain rungs

The table above says how many rungs are shorter than one refresh period; this says whether that
matters. The test is run **at the gain and the exposures the session will actually use**, because
flicker is not a property of the panel alone but of the panel, the exposure length and a rolling
shutter together.

It needs no knowledge of `g`, and it is not a mean-jitter test: a uniform level shift between two
frames cancels in a pair difference taken about its own mean, so the thing that would survive
into the PTC is *structure* - a slice of the refresh cycle that differs row by row. Two numbers
per rung, then:

- **`g_est = S / (var - R^2)`**, with `R` in counts from session 01's bias sweep. If the panel is
  steady this is the same number at every exposure - it is `g` - and if short exposures carry an
  extra variance it sags at the short end.
- **the row ratio**: the scatter of row means of the pair difference over what that scatter would
  be if the pixels were independent. One means no row structure; above one means something
  scanned while the shutter was open.

A sag at the short end refutes "one level for all eight gains" and sends the session to a per-gain
level (see the argument above). A flat table licenses the whole ladder.

**A rung within 4 sigma of the ceiling is excluded from that verdict.** At gain 450 one frame's
own noise is hundreds of counts, so a bright rung clips its own shot-noise tail, the pair variance
comes in low, and `g_est` reads high - a variance deficit that has nothing to do with the panel.
The headroom column says how many sigmas of margin each rung has.


In [ ]:
FLICKER_GAIN = GAINS[-1]
PROBE_RUNGS = [0, 2, 4, 6, 9, 11]     # faintest to top, spanning sub- to multi-refresh


def read_noise_counts(gain, offset=OFFSET):
    """Session 01's measured R at the nearest swept gain, in ADC counts."""
    rows = np.genfromtxt(RESULTS / "bias_sweep.csv", delimiter=",", names=True)
    rows = rows[rows["offset"] == offset]
    return float(rows["R_at_offset"][np.argmin(abs(rows["gain"] - gain))])


def pair_stats(exposure_s, n_pairs=3):
    """Signal, pair-difference variance and row structure at one exposure.

    Rule 2 of the protocol's analysis rules, run early and on one plane: the
    variance of a frame pair's difference, halved.  Two diagnostics come free
    from the same frames.  The row ratio is what a scanning panel or a backlight
    switching mid-readout shows up in, because neither is uniform over rows.
    The mean CV is the loudest PWM canary: a backlight that is off for part of
    a sub-refresh exposure moves the whole frame, and while a uniform move
    largely cancels in the pair difference it is the thing to see before
    trusting a dimmed panel.
    """
    asi.capture(rig, exposure_s, imagetyp="LIGHT")            # discard after the change
    sig, var, rows, frames = [], [], [], []
    for _ in range(n_pairs):
        a, _ = asi.capture(rig, exposure_s, imagetyp="LIGHT")
        b, _ = asi.capture(rig, exposure_s, imagetyp="LIGHT")
        pa = stats.to_adc(spatial.split(a)[bright]).astype(float)
        pb = stats.to_adc(spatial.split(b)[bright]).astype(float)
        d = pa - pb
        frames += [pa.mean(), pb.mean()]
        sig.append(0.5 * (pa.mean() + pb.mean()) - pedestal(FLICKER_GAIN))
        var.append(d.var(ddof=1) / 2.0)
        rows.append(d.mean(axis=1).std(ddof=1) / (d.std(ddof=1) / np.sqrt(d.shape[1])))
    frames = np.array(frames)
    return np.mean(sig), np.mean(var), np.mean(rows), frames.std(ddof=1) / frames.mean() * 100


set_level(level)
asi.configure(rig, gain=FLICKER_GAIN, offset=OFFSET, roi=ROI)
asi.capture(rig, lad[FLICKER_GAIN][0], imagetyp="LIGHT")      # 2 discards after a gain change
asi.capture(rig, lad[FLICKER_GAIN][0], imagetyp="LIGHT")
R = read_noise_counts(FLICKER_GAIN)

print(f"gain {FLICKER_GAIN}, grey level {level}, R = {R:.3f} counts, "
      f"one refresh period = {period * 1e3:.1f} ms\n")
head450 = FULL_SCALE - pedestal(FLICKER_GAIN)
print(f"{'exposure':>11} {'periods':>9} {'signal':>9} {'pair var':>10} "
      f"{'g_est':>9} {'row ratio':>10} {'mean CV':>9} {'headroom':>9}")
probe = []
for i in PROBE_RUNGS:
    e = lad[FLICKER_GAIN][i]
    if e < MIN_EXPOSURE:
        print(f"{e * 1e6:9.0f} us  below the {MIN_EXPOSURE * 1e6:.0f} us shutter floor -- skipped")
        continue
    S, var, row, cv = pair_stats(e)
    g_est = S / max(var - R ** 2, 1e-9)
    margin = (head450 - S) / np.sqrt(var)          # in sigmas of one frame's own noise
    probe.append((e, g_est, row, cv, margin))
    print(f"{e * 1e3:9.3f} ms {e / period:9.3f} {S:9.1f} {var:10.1f} "
          f"{g_est:9.4f} {row:10.3f} {cv:8.3f}% {margin:8.1f}s"
          f"{'  <- tail clipping, excluded' if margin < 4 else ''}")

clean = [r for r in probe if r[4] >= 4]          # a clipped tail is a variance deficit,
short = [g for e, g, _, _, _ in clean if e < period]      # not a flicker signal
long_ = [g for e, g, _, _, _ in clean if e >= period]
if short and long_:
    sag = np.mean(short) / np.mean(long_)
    print(f"\nsub-refresh g_est / multi-refresh g_est = {sag:.3f}")
    print("flat to a few percent: the ladder is licensed as it stands." if abs(sag - 1) < 0.05
          else "SAGGING: the short rungs carry a variance the long ones do not.  One level for "
               "all eight gains is refuted -- dim the panel for the top gains and record why.")
print(f"worst row ratio {max(r for _, _, r, _, _ in probe):.3f} "
      f"(1.0 is independent pixels; a scanning panel reads high)")
cv_short = max((c for e, _, _, c, _ in probe if e < period), default=0.0)
cv_long = max((c for e, _, _, c, _ in probe if e >= period), default=0.0)
print(f"worst frame-to-frame mean CV: {cv_short:.3f}% sub-refresh, {cv_long:.3f}% above.")
print("A dimmed backlight is the case to watch: PWM puts percent-level scatter into the short"
      "\nexposures and none into the long ones, and it is the reason brightness may not be"
      "\nlowered without re-running this cell.")


### Record for the bench

The scout ends with a **configuration, not a constant**: the grey level chosen above, the
sheet count, and the flux that pair produced, written into the session record with the ambient temperature. The attenuation
is valid only while nobody moves the camera off the panel.

If the attenuation the bench can reach falls short of what `t_sat(0)` needs, the honest response
is not to fudge the ladder. It is to say which gains the source can support and shorten the gain
set at the top - and to record that gain 600 was dropped for want of light rather than quietly
shooting it with a ladder whose bottom rungs sit under the shutter floor.

In [ ]:
set_level(level)
rig.close()      # drops the cooler too, but the scout never turned it on
print("camera closed")